# MethylGPT: Disease Risk & Survival Prediction

This notebook demonstrates two killer-application demos for using MethylGPT embeddings
in downstream predictive tasks:

**Demo 1 -- Age Prediction from Embeddings (AltumAge)**  
Uses the publicly available [AltumAge](https://github.com/rsinghlab/AltumAge) dataset to
show the full pipeline: extract embeddings, train Ridge regression on age, and evaluate
with Pearson *r*, MAE, R\u00b2, and AUC (binary above/below-median classification).

**Demo 2 -- Survival Prediction with C-Index**  
Shows the exact Ridge CV + concordance-index pipeline used in the MethylGPT disease-risk
study (8 disease categories, ~13 k train / ~3 k test). Demonstrates how to adapt the
framework for your own survival data, including multi-disease evaluation and Kaplan--Meier
visualisation.

**Requirements:**
- Pretrained MethylGPT model checkpoint
- CpG probe IDs CSV (`probe_ids_type3.csv`)
- AltumAge data (downloaded via `tutorials/finetuning_age_prediction/download_files.sh`)
- For Demo 2: your own embeddings with survival labels, *or* pre-extracted disease embeddings

In [ ]:
# Colab / environment setup
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q methylgpt[tutorials]
    !pip install -q gdown lifelines
    import torch

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: No GPU. Go to Runtime > Change runtime type > GPU")

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV, RidgeClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    roc_auc_score,
    roc_curve,
    accuracy_score,
)
from scipy.stats import pearsonr

try:
    from lifelines.utils import concordance_index
    from lifelines import KaplanMeierFitter
except ImportError:
    print("Install lifelines for survival analysis: pip install lifelines")

from methylgpt import MethylGPTModel, MethylVocab, create_dataloader
from methylgpt.inference import extract_embeddings

warnings.filterwarnings("ignore", message=".*IProgress.*")
warnings.filterwarnings("ignore", message=".*flash_attn.*")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load Model

Update the paths below to point to your pretrained MethylGPT model.  
See the [README](../../README.md#pretrained-models) for download links.

In [ ]:
# === UPDATE THESE PATHS ===
MODEL_DIR = Path("pretrained_models/methylgpt-medium")
CPG_LIST_FILE = "data/probe_ids_type3.csv"

# On Colab: download model, probe IDs, and sample data
if IN_COLAB:
    import gdown, subprocess
    os.makedirs("data", exist_ok=True)

    # 1. Download pretrained model (medium)
    if not list(MODEL_DIR.glob("*.pt")) if MODEL_DIR.exists() else True:
        os.makedirs(str(MODEL_DIR), exist_ok=True)
        print("Downloading methylgpt-medium model...")
        gdown.download_folder(
            "https://drive.google.com/drive/folders/14M4wdS83el9PAgh9TdfjSCeEcDPbz34f",
            output=str(MODEL_DIR),
            quiet=True,
        )
        print(f"Model downloaded to {MODEL_DIR}")
    else:
        print(f"Model already exists in {MODEL_DIR}")

    # 2. Download probe_ids_type3.csv
    if not os.path.exists(CPG_LIST_FILE):
        print("Downloading probe_ids_type3.csv...")
        subprocess.run([
            "wget", "-q", "-O", CPG_LIST_FILE,
            "https://www.dropbox.com/scl/fi/2n6bx7j8v0aon0kwfsghp/probe_ids_type3.csv?rlkey=ly133xlce1xxjiku6tiski6qq&st=pig4e41h&dl=1"
        ], check=True)
        print(f"Probe IDs saved to {CPG_LIST_FILE}")
    else:
        print(f"Probe IDs already exist at {CPG_LIST_FILE}")

    # 3. Download sample parquet data (for embedding extraction in Demo 1)
    PARQUET_DIR = "data/processed_type3_parquet_shuffled"
    if not os.path.exists(PARQUET_DIR):
        print("Downloading sample parquet data (~2 GB)...")
        subprocess.run([
            "wget", "-q", "--show-progress", "-O", "data/parquet_data.tar.gz",
            "https://www.dropbox.com/scl/fi/bbs6sxlkpbx11rhyvdfto/processed_type3_parquet_shuffled.tar.gz?rlkey=s73utmumq6xldmv3y6kh9bz75&st=8pslwy2a&dl=1"
        ], check=True)
        subprocess.run(["tar", "-xzf", "data/parquet_data.tar.gz", "-C", "data/"], check=True)
        os.remove("data/parquet_data.tar.gz")
        print(f"Parquet data extracted to {PARQUET_DIR}")
    else:
        print(f"Parquet data already exists at {PARQUET_DIR}")

# Load model config
with open(MODEL_DIR / "args.json", "r") as f:
    config = json.load(f)

model_files = list(MODEL_DIR.glob("*.pt"))
assert model_files, f"No .pt files found in {MODEL_DIR}"
config["load_model"] = True
config["pretrained_file"] = str(model_files[0])
config["mask_ratio"] = 0  # no masking for inference

vocab = MethylVocab(
    probe_id_dir=CPG_LIST_FILE,
    pad_token="<pad>",
    special_tokens=["<pad>", "<cls>", "<eoc>"],
    save_dir=None,
)

model = MethylGPTModel.from_pretrained(config, vocab)
model.eval().to(device)
if device.type == "cuda":
    model.half()

print(f"Model loaded: {config['layer_size']}-dim, {config['nlayers']} layers")


---

## Demo 1: Age Prediction from Embeddings (AltumAge)

The [AltumAge](https://github.com/rsinghlab/AltumAge) dataset contains DNA methylation
arrays with chronological age labels. It ships with the finetuning tutorial and can be
downloaded via:

```bash
cd tutorials/finetuning_age_prediction
bash download_files.sh
```

We extract MethylGPT embeddings from the raw methylation values, then train a simple
Ridge regression to predict age -- demonstrating that the learned representations
capture biologically meaningful variation without any finetuning.

In [ ]:
# ------------------------------------------------------------------
# Load AltumAge data and extract embeddings
# ------------------------------------------------------------------
ALTUMAGE_DIR = Path("../finetuning_age_prediction/data/altumage_metadata")

train_df = pd.read_parquet(ALTUMAGE_DIR / "altumage_train.parquet")
test_df = pd.read_parquet(ALTUMAGE_DIR / "altumage_test.parquet")

print(f"AltumAge  --  Train: {len(train_df)}, Test: {len(test_df)}")
print(f"Age range: {train_df['age'].min():.1f} -- {train_df['age'].max():.1f} years")
print(f"Columns: {train_df.columns.tolist()}")

# Create data loaders from the 'data' column (methylation values)
# Each row's 'data' column holds the methylation beta-value vector.
train_loader = create_dataloader(
    [str(ALTUMAGE_DIR / "altumage_train.parquet")], batch_size=32
)
test_loader = create_dataloader(
    [str(ALTUMAGE_DIR / "altumage_test.parquet")], batch_size=32
)

# Extract embeddings
print("\nExtracting train embeddings...")
X_train, train_ids = extract_embeddings(model, train_loader, device=str(device))
print("Extracting test embeddings...")
X_test, test_ids = extract_embeddings(model, test_loader, device=str(device))

print(f"\nTrain embeddings: {X_train.shape}")
print(f"Test embeddings:  {X_test.shape}")

# Align age labels to extracted sample order
train_age = train_df.set_index("id").loc[train_ids, "age"].values
test_age = test_df.set_index("id").loc[test_ids, "age"].values

In [ ]:
# ------------------------------------------------------------------
# Ridge regression for age prediction
# ------------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

alphas = np.logspace(-2, 4, 50)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
ridge = RidgeCV(alphas=alphas, cv=kfold, scoring="neg_mean_squared_error")
ridge.fit(X_train_scaled, train_age)

y_pred_train = ridge.predict(X_train_scaled)
y_pred_test = ridge.predict(X_test_scaled)

# Metrics
mae_train = mean_absolute_error(train_age, y_pred_train)
mae_test = mean_absolute_error(test_age, y_pred_test)
r2_train = r2_score(train_age, y_pred_train)
r2_test = r2_score(test_age, y_pred_test)
r_train, _ = pearsonr(train_age, y_pred_train)
r_test, _ = pearsonr(test_age, y_pred_test)

print("Age Prediction Results (Ridge Regression on MethylGPT embeddings)")
print("=" * 60)
print(f"{'':>15s}  {'Train':>10s}  {'Test':>10s}")
print(f"{'Pearson r':>15s}  {r_train:>10.3f}  {r_test:>10.3f}")
print(f"{'MAE (years)':>15s}  {mae_train:>10.2f}  {mae_test:>10.2f}")
print(f"{'R-squared':>15s}  {r2_train:>10.3f}  {r2_test:>10.3f}")
print(f"\nBest alpha: {ridge.alpha_:.4f}")

# --- Scatter plot: predicted vs. actual age ---
from aquarel import load_theme

theme = (
    load_theme("scientific")
    .set_grid(draw=False)
    .set_font(size=15)
    .set_ticks(direction="out")
    .set_axis_labels(pad=10)
)
theme.apply()

fig, ax = plt.subplots(figsize=(6, 6))

# Outline layer
ax.scatter(test_age, y_pred_test, s=30, c="black", alpha=1, zorder=1)
# Data layer
ax.scatter(test_age, y_pred_test, s=20, c="steelblue", alpha=0.7, zorder=2)

lims = [
    min(test_age.min(), y_pred_test.min()) - 2,
    max(test_age.max(), y_pred_test.max()) + 2,
]
ax.plot(lims, lims, "r--", alpha=0.8, label="y = x")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Chronological Age (years)")
ax.set_ylabel("Predicted Age (years)")
ax.set_title(f"Age Prediction  (r = {r_test:.2f}, MAE = {mae_test:.1f} yr)")
ax.legend(frameon=False)

theme.apply_transforms()

plt.savefig("age_prediction_scatter.pdf", bbox_inches="tight")
plt.savefig("age_prediction_scatter.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Binary classification: above / below median age  -->  AUC
# ------------------------------------------------------------------
median_age = np.median(train_age)
y_train_bin = (train_age > median_age).astype(int)
y_test_bin = (test_age > median_age).astype(int)

clf = RidgeClassifierCV(
    alphas=np.logspace(-3, 3, 20),
    cv=5,
)
clf.fit(X_train_scaled, y_train_bin)

y_score = clf.decision_function(X_test_scaled)
auc = roc_auc_score(y_test_bin, y_score)
acc = accuracy_score(y_test_bin, clf.predict(X_test_scaled))

print(f"Binary classification (median age = {median_age:.1f} years)")
print(f"  Accuracy : {acc:.3f}")
print(f"  AUC      : {auc:.3f}")
print(f"  Best alpha: {clf.alpha_:.4f}")

# --- ROC curve ---
theme.apply()

fpr, tpr, _ = roc_curve(y_test_bin, y_score)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, linewidth=2, label=f"Ridge (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve -- Age Binary Classification")
ax.legend(frameon=False)

theme.apply_transforms()

plt.savefig("age_roc_curve.pdf", bbox_inches="tight")
plt.savefig("age_roc_curve.png", dpi=600, bbox_inches="tight")
plt.show()

---

## Demo 2: Survival Prediction with C-Index

In the MethylGPT disease-risk study we evaluated survival prediction on 8 disease
categories from the Generation Scotland cohort (~13 k train, ~3 k test per disease).
Each sample's embedding (256 dims: `emb_0` ... `emb_255`) is paired with a
time-to-event `duration` and a binary censoring indicator `event`.

The pipeline:
1. Standardise embedding features (`StandardScaler`)
2. Log-transform survival times: `y = log(duration + 1)`
3. Fit `RidgeCV` with 5-fold CV over `np.logspace(-2, 4, 50)` alphas
4. Evaluate with the **concordance index** (C-index), negating predictions so that
   higher predicted log-duration maps to *lower* risk

Below we show the complete code. You can plug in your own embeddings with survival
labels, or use the pre-extracted disease embeddings from the working repository.

### Data format

The pipeline expects a Parquet (or CSV) file with the following columns:

| Column | Description |
|--------|-------------|
| `dnam_id` | Sample identifier |
| `emb_0` ... `emb_255` | MethylGPT embedding dimensions |
| `duration` | Time-to-event (days, years, etc.) |
| `event` | 1 = event observed, 0 = censored |
| `age` | (optional) Covariate |
| `sex` | (optional) Covariate |

You can produce this by:
- Extracting embeddings with `methylgpt.inference.extract_embeddings()`
- Merging with a survival-label table by sample ID

In [ ]:
# ------------------------------------------------------------------
# Load embeddings with survival labels
# ------------------------------------------------------------------
# Choose ONE of the options below and uncomment the corresponding block.

# ----- Option A: Load pre-extracted embeddings (Parquet with survival info) -----
# This is the typical path if you already ran the embedding extraction pipeline
# and merged embeddings with clinical / survival metadata.
#
# EMBEDDING_FILE = "embeddings/methylGPT_embeddings_Cancers_train.parquet"
# emb_df = pd.read_parquet(EMBEDDING_FILE)
# emb_cols = [f"emb_{i}" for i in range(256)]
# X = emb_df[emb_cols].values
# y_duration = emb_df["duration"].values
# y_event = emb_df["event"].values
# print(f"Loaded {X.shape[0]} samples, {X.shape[1]} features")

# ----- Option B: Extract embeddings, then merge with survival labels -----
# Use this when you have raw Parquet methylation data and a separate labels file.
#
# PARQUET_DIR = "data/processed_type3_parquet_shuffled"
# parquet_files = sorted([
#     os.path.join(PARQUET_DIR, f)
#     for f in os.listdir(PARQUET_DIR) if f.endswith(".parquet")
# ])
# data_loader = create_dataloader(parquet_files, batch_size=32)
# embeddings, sample_ids = extract_embeddings(model, data_loader, device=str(device))
#
# # Build embedding DataFrame
# emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
# emb_df = pd.DataFrame(embeddings, columns=emb_cols)
# emb_df.insert(0, "dnam_id", sample_ids)
#
# # Merge with survival labels
# survival_df = pd.read_csv("survival_labels.csv")  # dnam_id, duration, event
# merged = emb_df.merge(survival_df, on="dnam_id", how="inner")
# X = merged[emb_cols].values
# y_duration = merged["duration"].values
# y_event = merged["event"].values
# print(f"Merged {X.shape[0]} samples with survival labels")

print("Uncomment one of the options above and set paths to your data.")
print("The cells below show the complete pipeline code.")

In [ ]:
# ------------------------------------------------------------------
# Ridge CV + C-index survival prediction pipeline
# ------------------------------------------------------------------
# This cell contains the full pipeline.  It will run once you have
# loaded X, y_duration, y_event from the cell above.


def survival_ridge_pipeline(
    X,
    y_duration,
    y_event,
    test_size=0.2,
    random_state=42,
    verbose=True,
):
    """Train Ridge CV and evaluate with concordance index.

    Parameters
    ----------
    X : ndarray, shape (n_samples, n_features)
        Embedding features.
    y_duration : ndarray, shape (n_samples,)
        Time-to-event (days).
    y_event : ndarray, shape (n_samples,)
        Event indicator (1 = event, 0 = censored).
    test_size : float
        Fraction of data for test set.
    random_state : int
        Random seed.
    verbose : bool
        Print results.

    Returns
    -------
    dict with keys: c_train, c_test, alpha, ridge, scaler,
                    y_pred_train, y_pred_test, split indices.
    """
    # Train / test split
    X_train, X_test, dur_train, dur_test, evt_train, evt_test = train_test_split(
        X, y_duration, y_event, test_size=test_size, random_state=random_state
    )

    # Feature scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Log-transform survival times
    y_train_log = np.log(dur_train + 1)

    # Ridge CV
    alphas = np.logspace(-2, 4, 50)
    kfold = KFold(n_splits=5, shuffle=True, random_state=random_state)
    ridge = RidgeCV(alphas=alphas, cv=kfold, scoring="neg_mean_squared_error")
    ridge.fit(X_train_scaled, y_train_log)

    # Predict and evaluate
    # NOTE: predictions are NEGATED for C-index because higher predicted
    # log-duration means lower risk, and concordance_index expects higher
    # values = higher risk when used as the "predicted" argument.
    y_pred_train = ridge.predict(X_train_scaled)
    y_pred_test = ridge.predict(X_test_scaled)

    c_train = concordance_index(dur_train, -y_pred_train, evt_train)
    c_test = concordance_index(dur_test, -y_pred_test, evt_test)

    if verbose:
        print(f"Train C-index: {c_train:.4f}")
        print(f"Test C-index:  {c_test:.4f}")
        print(f"Best alpha:    {ridge.alpha_:.4f}")

    return {
        "c_train": c_train,
        "c_test": c_test,
        "alpha": ridge.alpha_,
        "ridge": ridge,
        "scaler": scaler,
        "y_pred_train": y_pred_train,
        "y_pred_test": y_pred_test,
        "dur_train": dur_train,
        "dur_test": dur_test,
        "evt_train": evt_train,
        "evt_test": evt_test,
    }


# --- Run the pipeline (uncomment after loading data) ---
# results = survival_ridge_pipeline(X, y_duration, y_event)
print("Pipeline function defined.  Call survival_ridge_pipeline(X, y_duration, y_event) to run.")

In [ ]:
# ------------------------------------------------------------------
# Multi-disease evaluation loop + C-index bar chart
# ------------------------------------------------------------------
# When you have per-disease embedding files (one train + one test Parquet
# per disease), iterate over them and collect results.


def evaluate_multiple_diseases(disease_dir, diseases=None):
    """Evaluate Ridge CV + C-index across multiple disease categories.

    Expects files named:
        {disease_dir}/methylGPT_embeddings_{disease}_train.parquet
        {disease_dir}/methylGPT_embeddings_{disease}_test.parquet

    Parameters
    ----------
    disease_dir : str or Path
        Directory containing per-disease embedding Parquet files.
    diseases : list of str, optional
        Disease names.  If None, auto-detect from filenames.

    Returns
    -------
    pd.DataFrame with columns: disease, c_train, c_test, alpha, n_train, n_test
    """
    disease_dir = Path(disease_dir)
    emb_cols = [f"emb_{i}" for i in range(256)]

    if diseases is None:
        # Auto-detect from train files
        diseases = sorted(
            set(
                f.stem.replace("methylGPT_embeddings_", "").replace("_train", "")
                for f in disease_dir.glob("*_train.parquet")
            )
        )

    rows = []
    for disease in diseases:
        train_path = disease_dir / f"methylGPT_embeddings_{disease}_train.parquet"
        test_path = disease_dir / f"methylGPT_embeddings_{disease}_test.parquet"

        if not train_path.exists() or not test_path.exists():
            print(f"  Skipping {disease}: files not found")
            continue

        train_df = pd.read_parquet(train_path)
        test_df = pd.read_parquet(test_path)

        X_tr = train_df[emb_cols].values
        X_te = test_df[emb_cols].values
        dur_tr = train_df["duration"].values
        dur_te = test_df["duration"].values
        evt_tr = train_df["event"].values
        evt_te = test_df["event"].values

        # Fit
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        y_train_log = np.log(dur_tr + 1)
        alphas = np.logspace(-2, 4, 50)
        kfold = KFold(n_splits=5, shuffle=True, random_state=42)
        ridge = RidgeCV(alphas=alphas, cv=kfold, scoring="neg_mean_squared_error")
        ridge.fit(X_tr_s, y_train_log)

        c_tr = concordance_index(dur_tr, -ridge.predict(X_tr_s), evt_tr)
        c_te = concordance_index(dur_te, -ridge.predict(X_te_s), evt_te)

        rows.append(
            {
                "disease": disease,
                "c_train": c_tr,
                "c_test": c_te,
                "alpha": ridge.alpha_,
                "n_train": len(train_df),
                "n_test": len(test_df),
            }
        )
        print(f"  {disease:30s}  C-train={c_tr:.4f}  C-test={c_te:.4f}  (n={len(train_df)}+{len(test_df)})")

    return pd.DataFrame(rows)


# --- Example usage (uncomment and set path) ---
# DISEASE_DIR = "embeddings"  # directory with per-disease parquet files
# diseases = [
#     "Autoimmune", "Cancers", "Cardiovascular",
#     "Endocrine_and_Metabolic", "Kidney", "Neurological",
#     "Respiratory", "date_of_death",
# ]
# results_df = evaluate_multiple_diseases(DISEASE_DIR, diseases)
# results_df.to_csv("disease_cindex_results.csv", index=False)

print("Multi-disease evaluation function defined.")
print("Call evaluate_multiple_diseases(disease_dir, diseases) after setting paths.")

In [ ]:
# ------------------------------------------------------------------
# Visualisation:  C-index bar chart + Kaplan-Meier curves
# ------------------------------------------------------------------
# This cell produces publication-quality figures once you have results.
# Uncomment the data-loading lines and adjust as needed.


def plot_cindex_bar(results_df, save_prefix="cindex_barchart"):
    """Bar chart of test C-index across disease categories."""
    from aquarel import load_theme

    theme = (
        load_theme("scientific")
        .set_grid(draw=False)
        .set_font(size=15)
        .set_ticks(direction="out")
        .set_axis_labels(pad=10)
    )
    theme.apply()

    df = results_df.sort_values("c_test", ascending=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(df["disease"], df["c_test"], color="steelblue", edgecolor="black", height=0.6)
    ax.axvline(0.5, color="gray", linestyle="--", alpha=0.6, label="Random (C = 0.5)")

    # Annotate values
    for bar, val in zip(bars, df["c_test"]):
        ax.text(
            val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=11,
        )

    ax.set_xlabel("Test C-index")
    ax.set_title("MethylGPT Survival Prediction by Disease")
    ax.legend(frameon=False)

    theme.apply_transforms()

    plt.savefig(f"{save_prefix}.pdf", bbox_inches="tight")
    plt.savefig(f"{save_prefix}.png", dpi=600, bbox_inches="tight")
    plt.show()


def plot_kaplan_meier(dur_test, evt_test, y_pred_test, disease_name="Disease", save_prefix="km_curve"):
    """Kaplan-Meier curves stratified by predicted risk (high vs. low)."""
    from aquarel import load_theme

    theme = (
        load_theme("scientific")
        .set_grid(draw=False)
        .set_font(size=15)
        .set_ticks(direction="out")
        .set_axis_labels(pad=10)
    )
    theme.apply()

    # Stratify by median predicted risk (negated predictions)
    risk_score = -y_pred_test
    median_risk = np.median(risk_score)
    high_risk = risk_score >= median_risk
    low_risk = ~high_risk

    fig, ax = plt.subplots(figsize=(7, 5))
    kmf = KaplanMeierFitter()

    kmf.fit(dur_test[low_risk], evt_test[low_risk], label="Low risk")
    kmf.plot_survival_function(ax=ax, color="steelblue", linewidth=2)

    kmf.fit(dur_test[high_risk], evt_test[high_risk], label="High risk")
    kmf.plot_survival_function(ax=ax, color="coral", linewidth=2)

    ax.set_xlabel("Time")
    ax.set_ylabel("Survival Probability")
    ax.set_title(f"Kaplan-Meier Curves -- {disease_name}")
    ax.legend(frameon=False)

    theme.apply_transforms()

    plt.savefig(f"{save_prefix}.pdf", bbox_inches="tight")
    plt.savefig(f"{save_prefix}.png", dpi=600, bbox_inches="tight")
    plt.show()


# --- Example usage (uncomment after running pipeline) ---
# plot_cindex_bar(results_df)
# plot_kaplan_meier(
#     results["dur_test"], results["evt_test"], results["y_pred_test"],
#     disease_name="Cancers",
# )

print("Visualisation functions defined: plot_cindex_bar() and plot_kaplan_meier()")

---

## Adapting for Your Data

To apply this pipeline to your own cohort:

1. **Prepare methylation data** as Parquet files (one row per sample, columns =
   CpG beta values matching the `probe_ids_type3.csv` vocabulary).

2. **Extract embeddings**:
   ```python
   loader = create_dataloader(parquet_files, batch_size=32)
   embeddings, sample_ids = extract_embeddings(model, loader, device="cuda")
   ```

3. **Merge with clinical labels** (duration + event for survival, or any
   continuous / binary outcome):
   ```python
   emb_df = pd.DataFrame(embeddings, columns=[f"emb_{i}" for i in range(256)])
   emb_df["dnam_id"] = sample_ids
   merged = emb_df.merge(clinical_df, on="dnam_id")
   ```

4. **Run the pipeline**:
   - For **survival analysis**: use `survival_ridge_pipeline()` from this notebook.
   - For **classification**: use `RidgeClassifierCV` (as shown in Demo 1).
   - For **continuous regression**: use `RidgeCV` directly.

5. **Evaluate**:
   - Survival: concordance index (C-index)
   - Classification: AUC, accuracy
   - Regression: Pearson r, MAE, R-squared

## Next Steps

- [Quickstart](../quickstart/quickstart.ipynb) -- Basic MethylGPT usage
- [Extract Embeddings](../get_embeddings/get_embeddings.ipynb) -- Full embedding extraction pipeline
- [Age Prediction (Finetuning)](../finetuning_age_prediction/age_prediction.ipynb) -- End-to-end finetuning for age prediction
- [CpG Selection](../cpg_selection/cpg_selection.ipynb) -- Attention-based CpG site importance analysis
- [Imputation](../imputation/imputation.ipynb) -- Recover missing CpG values